
# XGBoost Nutrient-Aware Pathogen Suppression Predictor 0103

This is the strictest version of the nutrient-aware model.

Features used (41 total):
  - R0_c1  ... R0_c20   : initial nutrient concentrations (continuous)
  - active_c1 ... active_c20 : binary flags — is this nutrient present?
  - diversity_level      : how many carbon sources are active (1/2/4/8/16)

Features intentionally excluded:
  - Species_0002 ... Species_0101 : ALL species membership columns REMOVED
  - community_size                : REMOVED 



Input:  community_index_v8_backup.csv
Target: community_suppression (float in [0, 1])

# Run03



In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                     GroupKFold, GroupShuffleSplit)
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from scipy.stats import pearsonr, spearmanr
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import shap

In [2]:
df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Codes\MiCRM_Final_Dataset\community_index_v10.csv")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Target stats — mean: {df['community_suppression'].mean():.3f}  "
      f"std: {df['community_suppression'].std():.3f}  "
      f"range: [{df['community_suppression'].min():.3f}, {df['community_suppression'].max():.3f}]")

Dataset shape: 75,000 rows × 168 columns
Target stats — mean: 0.517  std: 0.333  range: [0.000, 0.999]


In [3]:
# feature matrix

r0_cols = [f"R0_c{k}" for k in range(1, 21)]

active_cols = [f"active_c{k}" for k in range(1, 21)]

diversity_col = ['diversity_level']


feature_cols = r0_cols + active_cols + diversity_col

print(f"Feature count:          {len(feature_cols)}  (vs 42 in nutrient+size, 142 in full model)")
print(f"  R0 concentrations:    {len(r0_cols)}")
print(f"  Active flags:         {len(active_cols)}")
print(f"  diversity_level:      {len(diversity_col)}")
print(f"  community_size:       0  ← removed (biological, not chemical)")
print(f"  Species membership:   0  ← removed")

# Diagnostic: confirm all feature columns exist in the loaded file
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    print(f"\nWARNING: {len(missing)} missing columns: {missing}")
else:
    print("\nAll feature columns confirmed present in file.")

X = df[feature_cols]
y = df['community_suppression']

# groups: the broth each ROW belongs to. Aligned with X (same order, same index).
# Every group-aware split below uses this so that whole broths stay together.
groups = df["broth_id"].values

print(f"\nX shape: {X.shape}    y shape: {y.shape}")

Feature count:          41  (vs 42 in nutrient+size, 142 in full model)
  R0 concentrations:    20
  Active flags:         20
  diversity_level:      1
  community_size:       0  ← removed (biological, not chemical)
  Species membership:   0  ← removed

All feature columns confirmed present in file.

X shape: (75000, 41)    y shape: (75000,)


In [5]:
strat_key = df["community_size"].astype(str) + "_" + df["diversity_level"].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key)

# broth label for each TRAIN row, positionally aligned with X_train.
# df.loc[X_train.index, ...] reorders the broth column to match X_train's rows.
groups_train = df.loc[X_train.index, "broth_id"].values


def inner_holdout(X_tr, y_tr, groups_tr, frac=0.15, seed=42):
    """Carve a group-aware validation set out of training data for early stopping.

    GroupShuffleSplit makes one split that puts whole broths (never split) into
    either the fit part or the validation part. We only need one validation set
    for early stopping, so one split is exactly right .
    .split(...) yields (fit_idx, val_idx) as POSITIONAL indices into X_tr, so we
    take them with .iloc.
    """
    gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    fit_idx, val_idx = next(gss.split(X_tr, y_tr, groups=groups_tr))
    return (X_tr.iloc[fit_idx], y_tr.iloc[fit_idx],
            X_tr.iloc[val_idx], y_tr.iloc[val_idx])




In [6]:
# LINEAR BASELINES 

print("Baseline comparisons")
y_mean      = y_train.mean()
y_pred_mean = np.full_like(y_test, fill_value=y_mean, dtype=float)
r2_mean     = r2_score(y_test, y_pred_mean)
print(f"Baseline (global mean predictor):  R2 = {r2_mean:.4f}")

# StandardScaler: subtract mean, divide by std, so every feature is on the same
# scale. fit_transform LEARNS the mean/std on train; transform REUSES them on test
# (never re-learning on test -> no leak).
scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

ridge_cv = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5, scoring="r2")
ridge_cv.fit(X_train_scaled, y_train)
y_pred_ridge = ridge_cv.predict(X_test_scaled)
r2_ridge  = r2_score(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
print(f"Ridge (alpha={ridge_cv.alpha_:.5f}):  R2 = {r2_ridge:.4f}  MAE = {mae_ridge:.4f}")

lasso_cv = LassoCV(cv=5, max_iter=10000, n_alphas=50, random_state=42, n_jobs=-1)
lasso_cv.fit(X_train_scaled, y_train)
y_pred_lasso = lasso_cv.predict(X_test_scaled)
r2_lasso  = r2_score(y_test, y_pred_lasso)
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
print(f"Lasso (alpha={lasso_cv.alpha_:.5f}):  R2 = {r2_lasso:.4f}  MAE = {mae_lasso:.4f}")

enet_cv = ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95], cv=5, max_iter=10000,
                       n_alphas=50, random_state=42, n_jobs=-1)
enet_cv.fit(X_train_scaled, y_train)
y_pred_enet = enet_cv.predict(X_test_scaled)
r2_enet  = r2_score(y_test, y_pred_enet)
mae_enet = mean_absolute_error(y_test, y_pred_enet)
print(f"ElasticNet (alpha={enet_cv.alpha_:.5f}, l1={enet_cv.l1_ratio_:.2f}):  "
      f"R2 = {r2_enet:.4f}  MAE = {mae_enet:.4f}")

# xgb initial


Xi_fit, yi_fit, Xi_val, yi_val = inner_holdout(X_train, y_train, groups_train)

model_init = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, min_child_weight=3, reg_lambda=1.0,
    random_state=42, n_jobs=-1, eval_metric="rmse",
    early_stopping_rounds=30, verbosity=0)
# eval_set is the INNER validation set (from training broths). X_test is absent.
model_init.fit(Xi_fit, yi_fit, eval_set=[(Xi_val, yi_val)], verbose=False)
y_pred_init = model_init.predict(X_test)
r2_init  = r2_score(y_test, y_pred_init)
mae_init = mean_absolute_error(y_test, y_pred_init)
print(f"\nXGBoost (initial):  R2 = {r2_init:.4f}  MAE = {mae_init:.4f}  "
      f"trees used = {model_init.best_iteration}")

Baseline comparisons
Baseline (global mean predictor):  R2 = -0.0000
Ridge (alpha=0.00100):  R2 = 0.5470  MAE = 0.1776
Lasso (alpha=0.00011):  R2 = 0.5470  MAE = 0.1777
ElasticNet (alpha=0.00012, l1=0.95):  R2 = 0.5470  MAE = 0.1777

XGBoost (initial):  R2 = 0.6587  MAE = 0.1406  trees used = 340


In [7]:
# hyperparameter tuning

param_grid = {                            # tuning was already performed in a test run of this final updated code, so using those params
    "max_depth":        [5],
    "learning_rate":    [0.1],
    "subsample":        [0.6],
    "colsample_bytree": [0.8],
}

# GroupKFold(3): 3 CV folds where no broth appears in two folds. There is no
# shuffle/random_state because it partitions by GROUP, not by random rows.
group_cv = GroupKFold(n_splits=3)

# base_estimator has NO early stopping and NO eval_set, so tuning cannot peek
# at anything. n_estimators kept low for speed; the winner is retrained bigger.
base_estimator = xgb.XGBRegressor(
    n_estimators=200, min_child_weight=3, reg_lambda=1.0,
    random_state=42, n_jobs=-1, eval_metric="rmse", verbosity=0)

grid_search = GridSearchCV(
    estimator=base_estimator, param_grid=param_grid, cv=group_cv,
    scoring="neg_mean_squared_error", n_jobs=-1, verbose=1, refit=True)
# The 3rd argument groups=groups_train is what makes GridSearchCV honour the
# broth grouping. Without it, GridSearchCV silently splits by random rows.
grid_search.fit(X_train, y_train, groups=groups_train)
best_params = grid_search.best_params_
print("\nBest parameters (group-aware CV):")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

Fitting 3 folds for each of 1 candidates, totalling 3 fits

Best parameters (group-aware CV):
  colsample_bytree: 0.8
  learning_rate: 0.1
  max_depth: 5
  subsample: 0.6
Best CV RMSE: 0.2630


In [8]:
# FINAL MODEL  

Xf_fit, yf_fit, Xf_val, yf_val = inner_holdout(X_train, y_train, groups_train)

model_final = xgb.XGBRegressor(
    **best_params, n_estimators=1000, min_child_weight=3, reg_lambda=1.0,
    random_state=42, n_jobs=-1, eval_metric="rmse",
    early_stopping_rounds=40, verbosity=0)
# eval_set[0] = fit set (for the "train" curve), eval_set[1] = inner val (early
# stop + "validation" curve). Both come from TRAINING broths. X_test is not here.
model_final.fit(Xf_fit, yf_fit,
                eval_set=[(Xf_fit, yf_fit), (Xf_val, yf_val)], verbose=50)

y_pred_final = model_final.predict(X_test)      # test touched ONCE, here
r2_final   = r2_score(y_test, y_pred_final)
mae_final  = mean_absolute_error(y_test, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_test, y_pred_final))
print(f"\nFinal (interpolation reference) R2 = {r2_final:.4f}  "
      f"MAE = {mae_final:.4f}  RMSE = {rmse_final:.4f}  trees = {model_final.best_iteration}")

print("\nModel Comparison")
print(f"{'Model':<30}{'R2':>9}{'MAE':>9}")
print("-" * 48)
for name, r2v, maev in [

        ("XGBoost (tuned)", r2_final, mae_final)]:
    mae_str = "N/A" if np.isnan(maev) else f"{maev:.4f}"
    print(f"{name:<30}{r2v:>9.4f}{mae_str:>9}")

gkf = GroupKFold(n_splits=5)
fold_r2, fold_mae, fold_rmse = [], [], []

print("\nGroupKFold evaluation (true held-out broths):")

for fold_num, (tr_idx, te_idx) in enumerate(
        gkf.split(X, y, groups=groups), start=1):

    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

    groups_tr = groups.iloc[tr_idx] if hasattr(groups, "iloc") else groups[tr_idx]
    groups_te = groups.iloc[te_idx] if hasattr(groups, "iloc") else groups[te_idx]

    # safety check: no broth overlap between train and test
    assert len(set(groups_tr).intersection(set(groups_te))) == 0

    held_broths = sorted(set(df.iloc[te_idx]["broth_id"]))
    held_divs = sorted(df.iloc[te_idx]["diversity_level"].unique())

    print(
        f"\nFold {fold_num}: broths={held_broths}  "
        f"diversity levels={held_divs}  n_test={len(X_te):,}"
    )

    # inner validation split from training broths only
    Xg_fit, yg_fit, Xg_val, yg_val = inner_holdout(
        X_tr, y_tr, groups_tr, frac=0.15, seed=42 + fold_num
    )

    m_fold = xgb.XGBRegressor(
        **best_params,
        n_estimators=1000,
        min_child_weight=3,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        eval_metric="rmse",
        early_stopping_rounds=40,
        verbosity=0
    )

    # important: X_te/y_te are NOT in eval_set
    m_fold.fit(
        Xg_fit, yg_fit,
        eval_set=[(Xg_fit, yg_fit), (Xg_val, yg_val)],
        verbose=False
    )

    yp_fold = m_fold.predict(X_te)

    r2_f = r2_score(y_te, yp_fold)
    mae_f = mean_absolute_error(y_te, yp_fold)
    rmse_f = np.sqrt(mean_squared_error(y_te, yp_fold))

    fold_r2.append(r2_f)
    fold_mae.append(mae_f)
    fold_rmse.append(rmse_f)

    print(
        f"  R²={r2_f:.4f}  MAE={mae_f:.4f}  "
        f"RMSE={rmse_f:.4f}  trees={m_fold.best_iteration}"
    )

print(f"\nGroupKFold summary across {gkf.n_splits} folds:")
print(f"  R²:   mean={np.mean(fold_r2):.4f}  std={np.std(fold_r2):.4f}")
print(f"  MAE:  mean={np.mean(fold_mae):.4f}  std={np.std(fold_mae):.4f}")
print(f"  RMSE: mean={np.mean(fold_rmse):.4f}  std={np.std(fold_rmse):.4f}")

print(f"\nCompare to stratified-split R²={r2_final:.4f}")
print(
    f"Drop: {r2_final - np.mean(fold_r2):.4f}  "
    "(row-wise interpolation advantage / held-out-broth generalization gap)"
)

[0]	validation_0-rmse:0.31086	validation_1-rmse:0.36599
[50]	validation_0-rmse:0.18429	validation_1-rmse:0.27293
[100]	validation_0-rmse:0.17954	validation_1-rmse:0.26616
[150]	validation_0-rmse:0.17922	validation_1-rmse:0.26471
[200]	validation_0-rmse:0.17920	validation_1-rmse:0.26429
[250]	validation_0-rmse:0.17919	validation_1-rmse:0.26397
[300]	validation_0-rmse:0.17919	validation_1-rmse:0.26379
[350]	validation_0-rmse:0.17919	validation_1-rmse:0.26312
[400]	validation_0-rmse:0.17919	validation_1-rmse:0.26307
[450]	validation_0-rmse:0.17919	validation_1-rmse:0.26324
[457]	validation_0-rmse:0.17919	validation_1-rmse:0.26334

Final (interpolation reference) R2 = 0.6620  MAE = 0.1398  RMSE = 0.1932  trees = 417

Model Comparison
Model                                R2      MAE
------------------------------------------------
XGBoost (tuned)                  0.6620   0.1398

GroupKFold evaluation (true held-out broths):

Fold 1: broths=['Broth_005', 'Broth_010', 'Broth_015', 'Broth_020

In [9]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import xgboost as xgb

gss = GroupShuffleSplit(
    n_splits=20,
    test_size=0.20,
    random_state=42
)

r2_scores = []
mae_scores = []
rmse_scores = []

for split_num, (tr_idx, te_idx) in enumerate(gss.split(X, y, groups=groups), start=1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

    groups_tr = groups.iloc[tr_idx] if hasattr(groups, "iloc") else groups[tr_idx]
    groups_te = groups.iloc[te_idx] if hasattr(groups, "iloc") else groups[te_idx]

    # Safety check: no broth overlap between train and test
    assert len(set(groups_tr).intersection(set(groups_te))) == 0

    Xg_fit, yg_fit, Xg_val, yg_val = inner_holdout(
        X_tr, y_tr, groups_tr, frac=0.15, seed=42 + split_num
    )

    m = xgb.XGBRegressor(
        **best_params,
        n_estimators=1000,
        min_child_weight=3,
        reg_lambda=1.0,
        random_state=42 + split_num,
        n_jobs=-1,
        eval_metric="rmse",
        early_stopping_rounds=40,
        verbosity=0
    )

    m.fit(
        Xg_fit, yg_fit,
        eval_set=[(Xg_fit, yg_fit), (Xg_val, yg_val)],
        verbose=False
    )

    yp = m.predict(X_te)

    r2 = r2_score(y_te, yp)
    mae = mean_absolute_error(y_te, yp)
    rmse = np.sqrt(mean_squared_error(y_te, yp))

    r2_scores.append(r2)
    mae_scores.append(mae)
    rmse_scores.append(rmse)

    print(
        f"Split {split_num:02d}: "
        f"R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}  "
        f"trees={m.best_iteration}"
    )

print("\nGroupShuffleSplit held-out-broth summary:")
print(f"  R²:   mean={np.mean(r2_scores):.4f}  std={np.std(r2_scores):.4f}")
print(f"  MAE:  mean={np.mean(mae_scores):.4f}  std={np.std(mae_scores):.4f}")
print(f"  RMSE: mean={np.mean(rmse_scores):.4f}  std={np.std(rmse_scores):.4f}")

Split 01: R²=0.0530  MAE=0.2747  RMSE=0.3405  trees=30
Split 02: R²=0.5528  MAE=0.1774  RMSE=0.2260  trees=133
Split 03: R²=0.4063  MAE=0.2028  RMSE=0.2456  trees=32
Split 04: R²=0.4141  MAE=0.1894  RMSE=0.2445  trees=75
Split 05: R²=0.4163  MAE=0.1879  RMSE=0.2365  trees=164
Split 06: R²=0.4896  MAE=0.1773  RMSE=0.2277  trees=208
Split 07: R²=0.4718  MAE=0.1999  RMSE=0.2380  trees=37
Split 08: R²=0.3948  MAE=0.1932  RMSE=0.2540  trees=82
Split 09: R²=0.4302  MAE=0.2037  RMSE=0.2432  trees=19
Split 10: R²=0.3758  MAE=0.2005  RMSE=0.2611  trees=129
Split 11: R²=0.3921  MAE=0.1890  RMSE=0.2470  trees=148
Split 12: R²=0.4588  MAE=0.1866  RMSE=0.2454  trees=146
Split 13: R²=0.3936  MAE=0.2112  RMSE=0.2581  trees=51
Split 14: R²=0.2599  MAE=0.2102  RMSE=0.2796  trees=134
Split 15: R²=0.4353  MAE=0.1989  RMSE=0.2548  trees=88
Split 16: R²=0.6014  MAE=0.1863  RMSE=0.2195  trees=25
Split 17: R²=0.4327  MAE=0.2018  RMSE=0.2585  trees=186
Split 18: R²=0.4819  MAE=0.1835  RMSE=0.2282  trees=43
Sp

In [10]:
print("Performance by diversity level and community size")

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          y_pred_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

def condition_metrics(y_true_arr, y_pred_arr):
    y_t      = np.asarray(y_true_arr)
    y_p      = np.asarray(y_pred_arr)
    r2       = r2_score(y_t, y_p)
    mae      = mean_absolute_error(y_t, y_p)
    rmse     = np.sqrt(mean_squared_error(y_t, y_p))
    pr, _    = pearsonr(y_t, y_p)
    sr, _    = spearmanr(y_t, y_p)
    bias     = float(np.mean(y_p - y_t))
    std_true = float(np.std(y_t))
    return {'n': len(y_t), 'r2': r2, 'mae': mae, 'rmse': rmse,
            'pearson_r': pr, 'spearman_r': sr,
            'bias': bias, 'std_true': std_true}

hdr = f"{'':6} {'R²':>7} {'MAE':>7} {'RMSE':>7} {'Pearson':>8} {'Spearman':>9} {'Bias':>8} {'std(y)':>7} {'N':>6}"
print("\nBy diversity level:")
print(hdr)
print("-" * len(hdr))
for d, grp in results_df.groupby('diversity_level'):
    m    = condition_metrics(grp['y_true'], grp['y_pred'])
    flag = "  *" if m['std_true'] < 0.08 else "   "
    print(f"D={d:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}{flag}")
print("  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.")

print("\nBy community size:")
print(hdr)
print("-" * len(hdr))
for n, grp in results_df.groupby('community_size'):
    m = condition_metrics(grp['y_true'], grp['y_pred'])
    print(f"N={n:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}")

Performance by diversity level and community size

By diversity level:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
D=1     0.8488  0.0933  0.1684   0.9213    0.9375   0.0002  0.4330  3,000   
D=2     0.5595  0.1657  0.2366   0.7517    0.7774   0.0239  0.3564  3,000   
D=4     0.3917  0.1873  0.2296   0.6345    0.6220  -0.0266  0.2944  3,000   
D=8     0.5516  0.1400  0.1741   0.7428    0.7101  -0.0013  0.2600  3,000   
D=16    0.3470  0.1127  0.1392   0.5907    0.5608  -0.0040  0.1722  3,000   
  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.

By community size:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
N=12    0.4578  0.1748  0.2411   0.7801    0.7564   0.1267  0.3274  3,000
N=16    0.6252  0.1427  0.2026   0.8168    0.8039   0.0677  0.3309  3,000
N=24 

In [11]:
# shap analysis

print('shap analysis')

explainer = shap.TreeExplainer(model_final)
rng = np.random.default_rng(42)
idx = rng.choice(len(X_test), size=2000, replace=False)
shap_values = explainer.shap_values(X_test.iloc[idx])

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)

print("\nAll 41 features ranked by mean |SHAP|:")
print(shap_importance.round(5).to_string())

r0_shap     = shap_importance[r0_cols].sum()
active_shap = shap_importance[active_cols].sum()
div_shap    = shap_importance[diversity_col].sum()
total_shap  = r0_shap + active_shap + div_shap

print(f"\nSHAP by group:")
print(f"  R0 concentrations:  {r0_shap:.4f}  ({100*r0_shap/total_shap:.1f}%)")
print(f"  Active flags:       {active_shap:.4f}  ({100*active_shap/total_shap:.1f}%)")
print(f"  diversity_level:    {div_shap:.4f}  ({100*div_shap/total_shap:.1f}%)")

shap analysis

All 41 features ranked by mean |SHAP|:
R0_c8              0.05555
R0_c6              0.04006
R0_c9              0.02899
R0_c20             0.02634
R0_c16             0.02606
R0_c1              0.02424
active_c8          0.01873
R0_c3              0.01705
R0_c17             0.01603
R0_c10             0.01517
R0_c13             0.01391
R0_c2              0.01307
R0_c19             0.01216
R0_c7              0.01102
R0_c18             0.01077
R0_c14             0.01070
R0_c4              0.01026
R0_c11             0.00903
R0_c12             0.00810
R0_c5              0.00736
active_c6          0.00667
active_c14         0.00441
active_c9          0.00389
diversity_level    0.00331
R0_c15             0.00285
active_c12         0.00212
active_c19         0.00111
active_c2          0.00071
active_c17         0.00061
active_c11         0.00056
active_c5          0.00049
active_c20         0.00048
active_c7          0.00048
active_c15         0.00031
active_c16         0.00027
a

In [15]:
# figures


from matplotlib.patches import Patch

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(
    f'XGBoost PURE NUTRIENT (Chemistry-Only) Suppression Predictor'
    f'R²={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}  '
    f'[nutrient+size R²≈?  |  full model R²=0.930]',
    fontsize=12, fontweight='bold', y=0.99
)

# Panel 1: Predicted vs Actual coloured by diversity level
ax1 = fig.add_subplot(2, 3, 1)
for d, col in zip([1,2,4,8,16], ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3']):
    mask = df.loc[X_test.index, 'diversity_level'].values == d
    ax1.scatter(y_test.values[mask], y_pred_final[mask],
                alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.plot([0,1],[0,1],'r--',lw=1.5)
ax1.set_xlabel('True suppression')
ax1.set_ylabel('Predicted suppression')
ax1.set_title(f'Predicted vs Actual  R²={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)

# Panel 2: Residuals coloured by community size
# Even though size wasn't a feature, colouring by it reveals whether
# the model is systematically biased against certain community sizes.
ax2 = fig.add_subplot(2, 3, 2)
resid = y_test.values - y_pred_final
for n, col in zip([12,16,24,32,48], ['#1b9e77','#d95f02','#7570b3','#e7298a','#66a61e']):
    mask = df.loc[X_test.index, 'community_size'].values == n
    ax2.scatter(y_pred_final[mask], resid[mask],
                alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted suppression')
ax2.set_ylabel('Residual (true − predicted)')
ax2.set_title('Residuals (coloured by size — not a feature)')
ax2.legend(fontsize=7, markerscale=3)

# Panel 3: SHAP importance — all 41 features
ax3 = fig.add_subplot(2, 3, 3)
def feat_colour(f):
    if f in r0_cols:     return '#4CAF50'
    if f in active_cols: return '#FF9800'
    return '#9C27B0'   # diversity_level
top_n   = len(feature_cols)
top_shap = shap_importance.head(top_n)
cols = [feat_colour(f) for f in top_shap.index]
ax3.barh(range(top_n), top_shap.values[::-1], color=cols[::-1])
ax3.set_yticks(range(top_n))
ax3.set_yticklabels(top_shap.index[::-1], fontsize=6)
ax3.set_xlabel('Mean |SHAP value|')
ax3.set_title('All 41 Features by SHAP')
ax3.legend(handles=[
    Patch(fc='#4CAF50', label='R0 concentration'),
    Patch(fc='#FF9800', label='Active flag'),
    Patch(fc='#9C27B0', label='diversity_level'),
], loc='lower right', fontsize=8)

# Panel 4: R² by diversity level
ax4 = fig.add_subplot(2, 3, 4)
divs   = sorted(results_df['diversity_level'].unique())
r2_div = [r2_score(results_df[results_df['diversity_level']==d]['y_true'],
                   results_df[results_df['diversity_level']==d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level (# active nutrients)')
ax4.set_ylabel('R²')
ax4.set_title('R² by Diversity Level')
ax4.set_ylim(0, 1.05)
ax4.legend(fontsize=8)

# Panel 5: R² by community size
# Key diagnostic: if R² varies strongly by size, the model is missing
# the community_size signal that we deliberately excluded.
ax5 = fig.add_subplot(2, 3, 5)
sizes  = sorted(results_df['community_size'].unique())
r2_sz  = [r2_score(results_df[results_df['community_size']==n]['y_true'],
                   results_df[results_df['community_size']==n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N (NOT a feature in this model)')
ax5.set_ylabel('R²')
ax5.set_title('R² by Community Size\n(size was excluded — shows what we lost)')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=8)

# Panel 6: SHAP group pie
ax6 = fig.add_subplot(2, 3, 6)
ax6.pie(
    [r0_shap, active_shap, div_shap],
    labels=['R0\nconcentrations', 'Active\nflags', 'diversity_level'],
    colors=['#4CAF50', '#FF9800', '#9C27B0'],
    autopct='%.1f%%', startangle=90, textprops={'fontsize': 10}
)
ax6.set_title('SHAP Importance by Feature Group\n(pure nutrient model)')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('suppression_results_xgb03.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: suppression_results_xgb03.png")

Saved: suppression_results_xgb03.png


In [16]:
# save model

model_final.save_model('xgb03_model.json')
pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          y_pred_final,
    'residual':        y_test.values - y_pred_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
}).to_csv('predictions_xgb03.csv', index=False)
print("Saved")

Saved
